In [1]:
from __future__ import annotations
from pathlib import Path
import shutil
import sys
import yaml
from ultralytics import YOLO

# =========================
# 1) CONFIG
# =========================
# Choose ONE dataset yaml:
# Option A: merged dataset (recommended for your main project)
DATA_YAML = Path(r"C:/Users/foowe/Documents/Yolo/merged_dataset/data.yaml")

# Option B: single RAVDESS dataset
# DATA_YAML = Path(r"C:/Users/foowe/Documents/Yolo/ravdess_yolo/data.yaml")

MODEL_NAME = "yolov8s.pt"          # safer than yolov8l.pt for RTX 3070 while still stronger than n
EPOCHS = 100
IMGSZ = 640
BATCH = 8                           # keep this conservative first
DEVICE = 0                          # GPU 0
WORKERS = 0                         # Windows/Jupyter-safe
PATIENCE = 20                       # allow early stopping if plateau
LR0 = 0.001
OPTIMIZER = "SGD"                  # more stable baseline for your current project
CACHE = False                       # start safe; set True later if RAM allows
PROJECT_DIR = Path(r"C:/Users/foowe/Documents/Yolo/training_runs")
RUN_NAME = "emotion_detect_final"
FINAL_MODEL_PATH = Path(r"C:/Users/foowe/Documents/Yolo/final_models/emotion_detect_final_best.pt")

# Light augmentation only. Your earlier code was too aggressive for expression detection.
AUGMENT_KWARGS = dict(
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.3,
    scale=0.3,
    fliplr=0.5,
    flipud=0.0,
    mosaic=0.3,
    mixup=0.0,
)


# =========================
# 2) HELPERS
# =========================
def validate_data_yaml(data_yaml: Path) -> dict:
    if not data_yaml.exists():
        raise FileNotFoundError(f"data.yaml not found: {data_yaml}")

    with open(data_yaml, "r", encoding="utf-8") as f:
        data = yaml.safe_load(f)

    required_keys = ["train", "val", "nc", "names"]
    missing = [k for k in required_keys if k not in data]
    if missing:
        raise ValueError(f"data.yaml missing keys: {missing}")

    dataset_root = Path(data.get("path", data_yaml.parent))
    train_path = dataset_root / data["train"] if not Path(data["train"]).is_absolute() else Path(data["train"])
    val_path = dataset_root / data["val"] if not Path(data["val"]).is_absolute() else Path(data["val"])

    if not train_path.exists():
        raise FileNotFoundError(f"train path does not exist: {train_path}")
    if not val_path.exists():
        raise FileNotFoundError(f"val path does not exist: {val_path}")

    if int(data["nc"]) != len(data["names"]):
        raise ValueError(f"nc ({data['nc']}) does not match len(names) ({len(data['names'])})")

    return data


def export_best_model(save_dir: Path, final_model_path: Path) -> Path:
    best_pt = save_dir / "weights" / "best.pt"
    if not best_pt.exists():
        raise FileNotFoundError(f"Training finished but best.pt not found: {best_pt}")

    final_model_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_pt, final_model_path)
    return final_model_path


# =========================
# 3) MAIN TRAINING
# =========================
def main() -> int:
    print("=" * 70)
    print("YOLOv8 Final Training Script")
    print("=" * 70)
    print(f"Dataset YAML : {DATA_YAML}")
    print(f"Model        : {MODEL_NAME}")
    print(f"Epochs       : {EPOCHS}")
    print(f"Image Size   : {IMGSZ}")
    print(f"Batch        : {BATCH}")
    print(f"Device       : {DEVICE}")
    print(f"Workers      : {WORKERS}")
    print()

    data = validate_data_yaml(DATA_YAML)
    print("Dataset check passed.")
    print(f"Classes ({data['nc']}): {data['names']}")
    print()

    model = YOLO(MODEL_NAME)

    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        patience=PATIENCE,
        lr0=LR0,
        optimizer=OPTIMIZER,
        project=str(PROJECT_DIR),
        name=RUN_NAME,
        exist_ok=True,
        cache=CACHE,
        **AUGMENT_KWARGS,
    )

    save_dir = Path(results.save_dir)
    print(f"Training output dir: {save_dir}")

    exported = export_best_model(save_dir, FINAL_MODEL_PATH)
    print(f"Exported best model: {exported}")

    # quick validation pass
    trained_model = YOLO(str(exported))
    metrics = trained_model.val(data=str(DATA_YAML), split="val")
    print(f"mAP50    : {metrics.box.map50:.4f}")
    print(f"mAP50-95 : {metrics.box.map:.4f}")

    print("\nTraining pipeline completed successfully.")
    return 0


if __name__ == "__main__":
    try:
        raise SystemExit(main())
    except Exception as e:
        print(f"\nERROR: {e}")
        raise

YOLOv8 Final Training Script
Dataset YAML : C:\Users\foowe\Documents\Yolo\merged_dataset\data.yaml
Model        : yolov8s.pt
Epochs       : 100
Image Size   : 640
Batch        : 8
Device       : 0
Workers      : 0

Dataset check passed.
Classes (6): ['anger', 'cry', 'sad', 'happy', 'fear', 'neutral']

New https://pypi.org/project/ultralytics/8.4.38 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.10.20 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3070, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\foowe\Documents\Yolo\merged_dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=

SystemExit: 0

C:\Users\foowe\anaconda3\envs\yolo_rt\lib\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
